# tf.data によるデータ読み込みと前処理の最適化

**学習目標**
1. tf.data を使用してメモリからデータを読み込む方法を学ぶ
1. tf.data を使用してディスクからデータを読み込む方法を学ぶ
1. 特徴量エンジニアリング（バッチ処理、シャッフルなど）を含む本番環境向けの入力パイプラインを作成する方法を学ぶ
1. tf.data を使用してパイプラインを最適化する方法を学ぶ


このノートブックでは、前のラボで実装した線形回帰をリファクタリングして `tf.data.Dataset` からデータを取得するようにし、それを用いて**確率的勾配降下法（stochastic gradient descent）**を実装する方法を学習します。この場合、元のデータセットは合成データであり、`tf.data` API によってメモリから直接読み込まれます。

フレームワークには TensorFlow を使用しますが、**tf.data は JAX や PyTorch などの任意のフレームワークでも動作します**。

後半では、データセットがディスク上にある場合に `tf.data` API を使用してデータセットを読み込む方法を学び、データ パイプラインを最適化する方法を学習します。

In [ ]:
import os
import warnings

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")

import json
import math
from pprint import pprint

import numpy as np
import tensorflow as tf

print(tf.version.VERSION)

## メモリからのデータの読み込み

### データセットの作成

前のセクションの合成データセットを考えてみましょう:

In [ ]:
N_POINTS = 10
X = tf.constant(range(N_POINTS), dtype=tf.float32)
Y = 2 * X + 10

まず、以下を入力として受け取る関数を実装します。

- 線形関数 $y= 2x + 10$ によって生成された合成データの $X$ および $Y$ ベクトル
- トレーニングでデータセットを反復処理する回数（`epochs`）
- データセット内のバッチサイズ（`batch_size`）
そして `tf.data.Dataset` を返します: 

**備考:** データセットの要素を使い切ったため、最後のバッチには指定した正確な数の要素が含まれていない可能性があることに注意してください。

各バッチにまったく同じ数の要素を含めたい場合は、次のように設定して最後のバッチを破棄する必要があります。

```python
dataset = dataset.batch(batch_size, drop_remainder=True)
```

ここではそのように設定します。

In [ ]:
def create_dataset(X, Y, epochs, batch_size):
    dataset = tf.data.Dataset.from_tensor_slices((X, Y))
    dataset = dataset.repeat(epochs).batch(batch_size, drop_remainder=True)
    return dataset

3 つのデータポイントのバッチでデータセットを 2 回反復処理して、関数をテストしてみましょう:

In [ ]:
BATCH_SIZE = 3
EPOCH = 2

dataset = create_dataset(X, Y, epochs=EPOCH, batch_size=BATCH_SIZE)

for i, (x, y) in enumerate(dataset):
    print("x:", x.numpy(), "y:", y.numpy())
    assert len(x) == BATCH_SIZE
    assert len(y) == BATCH_SIZE
assert EPOCH

### 損失関数と勾配

損失関数と勾配を計算する関数は前と同じです:

In [ ]:
def loss_mse(X, Y, w0, w1):
    Y_hat = w0 * X + w1
    errors = (Y_hat - Y) ** 2
    return tf.reduce_mean(errors)


def compute_gradients(X, Y, w0, w1):
    with tf.GradientTape() as tape:
        loss = loss_mse(X, Y, w0, w1)
    return tape.gradient(loss, [w0, w1])

### トレーニング ループ

現在との主な違いは、トレーニング ループ内で、`create_dataset` 関数によって生成された `tf.data.Dataset` を直接イテレートすることです。 

2 のバッチサイズで合成データセットを 250 回反復するようにデータセットを設定します。

In [ ]:
EPOCHS = 250
BATCH_SIZE = 2
LEARNING_RATE = 0.02

MSG = "STEP {step} - loss: {loss}, w0: {w0}, w1: {w1}\n"

w0 = tf.Variable(0.0)
w1 = tf.Variable(0.0)

dataset = create_dataset(X, Y, epochs=EPOCHS, batch_size=BATCH_SIZE)

for step, (X_batch, Y_batch) in enumerate(dataset):
    dw0, dw1 = compute_gradients(X_batch, Y_batch, w0, w1)
    w0.assign_sub(dw0 * LEARNING_RATE)
    w1.assign_sub(dw1 * LEARNING_RATE)

    if step % 100 == 0:
        loss = loss_mse(X_batch, Y_batch, w0, w1)
        print(MSG.format(step=step, loss=loss, w0=w0.numpy(), w1=w1.numpy()))

assert loss < 0.0001
assert abs(w0 - 2) < 0.001
assert abs(w1 - 10) < 0.001

## ディスクからのデータの読み込み

### CSV ファイルの配置の確認

前のラボで書き出した **taxifare データセット** の CSV ファイルから始めます。 

taxifare データセット ファイルは `../data` に保存されています。

下のセルでファイルが存在することを確認してください。存在しない場合は、前のラボのノートブックを実行して taxifare データセットを再生成してください:

In [ ]:
!ls -l ../data/taxi*.csv

### 低レベル tf.data API を使用した CSV ファイルの読み込み

より柔軟なパイプラインを構築するために、低レベルの tf.data API を活用してパイプラインの動作を完全に制御できます。

CSV を含むテキストベースのデータには、`TextLineDataset` を使用してデータを読み込むことができます。

In [ ]:
ds = tf.data.TextLineDataset("../data/taxi-train.csv")
ds

Dataset オブジェクト（`ds`）はまだ単なる定義であり、実際のデータはまだ読み込まれていないことに注意してください。<br>
`dataset.take(2)` を使用して、このデータセットの最初の 2 つの要素を反復処理してみましょう:

In [ ]:
for data in ds.take(2):
    print(data)

最初の要素としてヘッダー行が読み込まれているようです。これはトレーニング データの一部ではないため、`skip()` メソッドでスキップしましょう。

In [ ]:
ds = tf.data.TextLineDataset("../data/taxi-train.csv").skip(1)

for data in ds.take(2):
    print(data)

### `.map()` による特徴量の変換

現時点では、CSV ファイルをテキスト ファイルとして読み込んでおり、各行は区切り文字（`,`）を含む単一の文字列値として表されているだけです。

行を受け取り、それを複数の値に分割するパース関数を作成しましょう。

In [ ]:
def parse_csv(row):
    return tf.strings.split(row, ",")

for ループ内で `.take()` とともにこの関数を呼び出して、正しく動作することを確認しましょう。 

In [ ]:
ds = tf.data.TextLineDataset("../data/taxi-train.csv").skip(1)

for data in ds.take(2):
    values = parse_csv(data)
    pprint(values)

for ループ内で関数を呼び出す代わりに、`.map()` メソッドでラップしてパイプラインに組み込むことができます。

In [ ]:
ds = tf.data.TextLineDataset("../data/taxi-train.csv").skip(1).map(parse_csv)

for data in ds.take(2):
    print(data)

それでは、`parse_csv` 関数を拡張しましょう。<br>
機械学習のトレーニングでは、トレーニング データを `(features, label)` のタプルとして渡す必要があります。

この CSV ファイルには次の列があります:

In [ ]:
!head -1 ../data/taxi-train.csv

特徴量として `pickuplon`、`pickuplat`、`dropofflon`、`dropofflat` を使用して、`fare_amount` の値を予測したいとします。その場合は次のようになります:

In [ ]:
def parse_csv(row):
    columns = tf.strings.split(row, ",")
    # ラベル: fare_amount
    label = tf.strings.to_number(columns[0])
    # 特徴量: pickup_longitude, pickup_latitude, dropoff_longitude, dropoff_latitude
    features = tf.strings.to_number(columns[2:6])
    return features, label

In [ ]:
ds = tf.data.TextLineDataset("../data/taxi-train.csv").skip(1)
ds = ds.map(parse_csv)

for features, label in ds.take(2):
    print(f"features: \n  {features}, \nlabel: \n  {label} \n++++")

### バッチ処理

通常、機械学習のトレーニング モジュールにはバッチ処理されたデータが必要です。パイプラインに `.batch(BATCH_SIZE)` を追加して、データをバッチ処理するようにリファクタリングしましょう。

In [ ]:
BATCH_SIZE = 4

ds = tf.data.TextLineDataset("../data/taxi-train.csv").skip(1)
ds = ds.map(parse_csv).batch(BATCH_SIZE)

for features, label in ds.take(2):
    print(f"features: \n  {features}, \nlabel: \n  {label} \n++++")

これでデータセットは*行*ではなく*バッチ*のイテレータになり、ニューラル ネットワークのミニバッチ トレーニングに適した形式になりました。

### シャッフルとリピート

複数のワーカーにわたってバッチ単位でディープ ラーニング モデルをトレーニングする場合、データをシャッフルすると役立ちます。これにより、異なるワーカーが入力ファイルの異なる部分を同時に処理することになり、ワーカー間での勾配の平均化が効果的になります。<br>
`.shuffle()` を使用してシャッフルを追加できます。ただし、`buffer_size` で指定されたシャッフル バッファはメモリ上に保存されるため、非常に大規模なデータセットの完全なシャッフルには適していないことに注意してください。

データ パイプラインを `create_dataset` 関数でラップして、その動作を制御し、データセットがトレーニングに使用される場合にのみデータをシャッフルできるようにしましょう。

関数に追加の引数 `mode` を導入して、データをシャッフルする必要がある場合（`mode == "train"`）とシャッフルすべきでない場合（`mode == "eval"`）を関数の内部で区別できるようにします。

また、トレーニング中にデータを無制限に読み込めるように `.repeat()` を追加します。 

In [ ]:
def create_dataset(pattern, batch_size, mode="eval"):
    ds = tf.data.TextLineDataset(pattern).skip(1)
    ds = ds.map(parse_csv).repeat()
    if mode == "train":
        ds = ds.shuffle(buffer_size=1000)
    ds = ds.batch(batch_size, drop_remainder=True)
    return ds

両方のモードで関数が正常に動作することを確認しましょう:

In [ ]:
# このセルを複数回実行して、結果が異なることを確認してください。
tempds = create_dataset("../data/taxi-train.csv", 2, "train")
print(list(tempds.take(1)))

In [ ]:
tempds = create_dataset("../data/taxi-valid.csv", 2, "eval")
print(list(tempds.take(1)))

## tf.data によるパフォーマンスの向上

データの読み込みと前処理フェーズのパフォーマンスを最大化することは、多くの機械学習のユースケースにおいて重要です。

`tf.data` は、パフォーマンスのボトルの原因に応じて、プロセスを最適化するさまざまな方法を提供します。<br>
いくつかのシナリオを見てみましょう。

比較のために、トレーニング アプリケーションのループをシミュレートする次の `benchmark` 関数を使用します。

In [ ]:
import time


def benchmark(dataset, num_epochs=2):
    start_time = time.perf_counter()
    for epoch_num in range(num_epochs):
        for sample in dataset:
            # トレーニング ステップの実行
            time.sleep(0.01)
    print("Execution time:", time.perf_counter() - start_time)

### ケース 1: 重い map 操作によるパフォーマンスのボトルネック 

特徴量変換の `.map()` は柔軟で便利ですが、前処理関数に重い処理が含まれている場合、このプロセスがパフォーマンスのボトルネックになる可能性があります。

パース関数にスリープ時間を追加して、そのケースをシミュレートしてみましょう。

In [ ]:
def heavy_parse_csv(row):
    columns = tf.strings.split(row, ",")
    label = tf.strings.to_number(columns[0])
    features = tf.strings.to_number(columns[2:6])

    # 重い前処理を実行...
    tf.py_function(lambda: time.sleep(0.001), [], ())

    return features, label

In [ ]:
def create_dataset(pattern, batch_size=128):
    ds = tf.data.TextLineDataset(pattern).skip(1)
    ds = ds.map(heavy_parse_csv)
    ds = ds.batch(batch_size, drop_remainder=True)
    return ds

In [ ]:
tempds = create_dataset("../data/taxi-train.csv")
benchmark(tempds)

フローは次のようになります。この場合、データの読み込みとトレーニングの間にある map 操作がボトルネックになっています。

![Map bottleneck](https://www.tensorflow.org/guide/images/data_performance/sequential_map.svg)

このプロセスを最適化する方法を見てみましょう。

#### 解決策 1: map の並列化

入力要素は互いに独立しているため、前処理を複数の CPU コアにわたって並列化できます。これを可能にするため、map 変換には並列処理のレベルを指定する num_parallel_calls 引数が用意されています。

`.map()` では、関数と共に `num_parallel_calls` 引数を指定できます。並列数は `tf.data.AUTOTUNE` を指定することで自動チューニングできます。

In [ ]:
def create_dataset(pattern, batch_size=128):
    ds = tf.data.TextLineDataset(pattern).skip(1)
    ds = ds.map(heavy_parse_csv, num_parallel_calls=tf.data.AUTOTUNE)
    ds = ds.batch(batch_size, drop_remainder=True)
    return ds

In [ ]:
tempds = create_dataset("../data/taxi-train.csv")
benchmark(tempds)

大幅に高速化されました！

現在のフローは次のようになります。
![parallelized](https://www.tensorflow.org/guide/images/data_performance/parallel_map.svg)

#### 解決策 2: map 操作のベクトル化
`.map()` は `Dataset` から返される個々の要素を処理します。現在の関数は一度に 1 つの CSV 要素を処理するように構成されています。しかし、実行可能であれば、データをバッチ単位で処理する方が常に効率的です。

関数をベクトル化（つまり、一度に入力のバッチに対して動作するように）し、`map` 変換の前に `batch` 変換を適用しましょう。

In [ ]:
def heavy_parse_csv_batch(row):
    columns = tf.strings.split(row, ",").to_tensor()
    label = tf.strings.to_number(columns[:, 0])
    features = tf.strings.to_number(columns[:, 2:6])

    # 重い前処理を実行...
    tf.py_function(lambda: time.sleep(0.001), [], ())

    return features, label

In [ ]:
def create_dataset(pattern, batch_size=128):
    ds = tf.data.TextLineDataset(pattern).skip(1)
    ds = ds.batch(batch_size, drop_remainder=True)
    ds = ds.map(heavy_parse_csv_batch, num_parallel_calls=tf.data.AUTOTUNE)
    return ds

In [ ]:
tempds = create_dataset("../data/taxi-train.csv")
benchmark(tempds)

### ケース 2: I/O（データ読み込み）のパフォーマンスのボトルネック 
次のシナリオを見てみましょう。

実際の環境では、入力データがリモート（たとえば別のリージョンにある Google Cloud Storage など）に保存されている場合があります。ローカルでデータを読み込むときにはうまく機能するデータセット パイプラインであっても、ローカル ストレージとリモート ストレージの以下のような違いにより、リモートからデータを読み込む際に I/O がボトルネックになる可能性があります。

- **最初の 1 バイトまでの時間（Time-to-first-byte）**: リモート ストレージからファイルの最初の 1 バイトを読み込むには、ローカル ストレージよりも桁違いに長い時間がかかる場合があります。
- **読み込みスループット（Read throughput）**: リモート ストレージは通常、大きな総帯域幅を提供しますが、単一のファイルを読み込む場合はこの帯域幅のほんの一部しか利用できない場合があります。

さらに、raw バイトがメモリに読み込まれたら、データのデシリアライズや復号（protobuf など）が必要になる場合もあり、これには追加の計算が必要です。このオーバーヘッドはデータがローカルまたはリモートのどちらに保存されているかに関係なく発生しますが、データが効果的にプリフェッチされていない場合、リモートのケースではさらに悪化する可能性があります。

このシナリオをシミュレートするカスタム データセットを作成してみましょう。

In [ ]:
class IOBoundDataset(tf.data.Dataset):
    def _generator(file_name, num_samples):
        # ファイルを開く
        time.sleep(0.3)

        for sample_idx in range(num_samples):
            # ファイルから各行を読み取る
            time.sleep(0.15)

            yield (sample_idx,)

    def __new__(cls, file_name, num_samples=5):
        return tf.data.Dataset.from_generator(
            cls._generator,
            output_signature=tf.TensorSpec(shape=(1,), dtype=tf.int64),
            args=(
                file_name,
                num_samples,
            ),
        )

In [ ]:
ds = IOBoundDataset("dummy_file.csv").repeat(20)
benchmark(ds)

#### 解決策 1: キャッシュ
機械学習のトレーニングでは同じデータセットを繰り返し使用することが多いため、毎回リモート ソースから再ロードするのではなく、最初のエポックでデータをキャッシュし、それ以降のエポックではキャッシュからデータを取得するのが優れた戦略です。

`.cache()` を挿入するだけでキャッシュを使用できます。

In [ ]:
ds = IOBoundDataset("dummy_file.csv").cache().repeat(20)
benchmark(ds)

はるかに高速になりました！ただし、キャッシュされたデータはメモリ上に保存されるため、何をキャッシュするかには注意してください。たとえば、テラバイト規模のデータセットをキャッシュすることは現実的ではありません。

たとえば、トレーニング用動画のパスを含む CSV ファイルがある場合、実際の動画ファイルをキャッシュする代わりに、動画読み込み関数の前に `cache()` を挿入して小さな CSV ファイルをキャッシュすることを検討してください。

```python 
dataset.map(parse_csv_fn).cache().map(load_video_fn)
```

#### 解決策 2: インターリーブ（Interleave）

さまざまなデータ抽出オーバーヘッドの影響を軽減するために、tf.data.Dataset.interleave 変換を使用してデータ読み込みステップを並列化し、他のデータセット（データ ファイル リーダーなど）のコンテンツをインターリーブ（交互に配置）させることができます。

複数のシャード化されたファイルがあるとします。

In [ ]:
files = [
    "dummy_file_shard001.csv",
    "dummy_file_shard002.csv",
    "dummy_file_shard003.csv",
]

複数のファイル読み込み操作をインターリーブするために `interleave()` を挿入できます。

In [ ]:
ds = tf.data.Dataset.from_tensor_slices(files)
ds = ds.interleave(IOBoundDataset)

benchmark(ds)

どのように読み込まれるかを見てみましょう。

In [ ]:
for d in ds:
    print(d)

各データセットには 0 から 4 までの値が含まれており、ここでは 3 つのファイルからのデータ読み込みがインターリーブされていることがわかります。

これはインターリーブのフロー図です（この場合は 2 つのファイル）。
![sequential interleave](https://www.tensorflow.org/guide/images/data_performance/sequential_interleave.svg)

さらにパフォーマンスを向上させるために、`.map()` と同様に、`num_parallel_calls` を追加してこのインターリーブ操作を並列化できます。

In [ ]:
ds = tf.data.Dataset.from_tensor_slices(files)
ds = ds.interleave(IOBoundDataset, num_parallel_calls=len(files))

benchmark(ds)

![parallel interleave](https://www.tensorflow.org/guide/images/data_performance/parallel_interleave.svg)

これらの手法を組み合わせることで、高度に最適化されたデータ読み込みおよび変換パイプラインを設計できます。

Copyright 2025 Google LLC

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

     https://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.